# LangChain 심화 : LCEL과 Runnable

이번 실습에서는 기초 파일에서 배운 Model, Prompt, Output Parser를 연결해 재사용 가능한 실행 흐름을 만듭니다.

**학습 목표**

> 1. 이번 실습에서는 LCEL(LangChain Expression Language)을 사용해 Prompt, Model, Output Parser를 하나의 실행 체인으로 연결하는 법을 익힌다.
> 2. `prompt | model | parser` 구조를 통해 데이터가 왼쪽에서 오른쪽으로 흐르는 LangChain 체인 구성 방식을 이해한다.
> 3. Runnable의 공통 실행 방식인 `invoke`, `batch`, `stream`을 사용해 단일 실행, 대량 실행, 스트리밍 실행을 연습한다.
> 4. **LCEL과 Runnable을 조립해 실무형 LLM 워크플로우를 만드는 법**을 익힌다.

# 1. 환경 준비

## (1) 라이브러리 설치

처음 실행하는 환경이라면 아래 셀의 주석을 해제하고 실행합니다. 이미 `pyproject.toml` 또는 `requirements.txt`로 설치했다면 실행하지 않아도 됩니다.

In [ ]:
# 필요한 라이브러리 설치
# %pip install -U langchain langchain-core langchain-openai python-dotenv pydantic pandas

## (2) 라이브러리 Import

이번 실습에서 사용하는 핵심 객체는 다음과 같습니다.

| 객체 | 역할 |
|---|---|
| `ChatOpenAI` | OpenAI 채팅 모델을 LangChain 방식으로 호출 |
| `PromptTemplate` | 문자열 기반 프롬프트 템플릿 |
| `ChatPromptTemplate` | system/human 메시지를 분리하는 채팅 프롬프트 |
| `StrOutputParser` | 모델 응답에서 문자열만 추출 |
| `RunnableLambda` | 일반 파이썬 함수를 체인에 연결 |
| `RunnableParallel` | 여러 체인을 동시에 실행 |


In [ ]:
import os
from pathlib import Path
from getpass import getpass
from typing import Literal

import pandas as pd
from dotenv import load_dotenv
from pydantic import BaseModel, Field


## (3) API Key 설정

API 키는 코드에 직접 작성하지 않습니다. 권장 방식은 `.env` 파일에 저장하는 것입니다.

```text
# OpenAI를 사용할 때
OPENAI_API_KEY=sk-...

# Gemini를 사용할 때
GOOGLE_API_KEY=...

```


In [ ]:
load_dotenv()

print("OPENAI_API_KEY:", "있음" if os.getenv("OPENAI_API_KEY") else "없음")
print("GOOGLE_API_KEY:", "있음" if os.getenv("GOOGLE_API_KEY") else "없음")

## (4) 모델 준비

심화 실습은 체인을 구성하는 데 집중하므로 모델은 바로 준비합니다.

# 2. Chain과 LCEL

- LCEL(LangChain Expression Language) : Runnable 객체를 `|` 파이프 연산자를 사용해 컴포넌트를 연결하고, 체인을 **선언적으로 정의**하는 표현 방식


이 구조에서 데이터는 왼쪽에서 오른쪽으로 흐릅니다.

| 단계 | 입력 | 출력 |
|---|---|---|
| Prompt | dict | PromptValue 또는 메시지 |
| Model | PromptValue 또는 메시지 | AIMessage |
| Parser | AIMessage | 문자열 또는 구조화 데이터 |

LCEL을 쓰면 프롬프트, 모델, 파서를 따로 실행하지 않고 하나의 실행 단위처럼 다룰 수 있습니다.

### 다이어그램

```mermaid
flowchart LR
    IN(["📥 dict input<br/>{role, question}"]):::input
    PT["📝 Prompt<br/>ChatPromptTemplate<br/>{role} · {question}"]:::node
    LM["🤖 LLM<br/>init_chat_model<br/>→ AIMessage"]:::node
    PA["🔍 Parser<br/>StrOutputParser<br/>→ str"]:::node
    OUT(["📤 최종 문자열"]):::output

    IN --> PT --> LM --> PA --> OUT

    classDef input  fill:#4f46e5,stroke:#3730a3,color:#fff
    classDef node   fill:#1e293b,stroke:#475569,color:#e2e8f0
    classDef output fill:#059669,stroke:#047857,color:#fff
```

코드 표현은 다음과 같이 간결합니다.

```
prompt | llm | parser
```

### 파이프 연산자의 이점

1. **가독성**: "프롬프트 → 모델 → 파서"의 처리 흐름이 코드에 그대로 드러납니다.
2. **재사용성**: `prompt | llm` 부분을 변수로 분리하면 서로 다른 파서와 조합할 수 있습니다.
3. **공통 인터페이스**: 각 Runnable은 `.invoke()`, `.stream()`, `.batch()`를 자동 지원합니다.

## LCEL을 쓰는 이유

LCEL의 장점은 단순히 코드가 짧아지는 것이 아닙니다. 모든 체인이 공통 실행 인터페이스를 가지게 됩니다.

| 실행 방식 | 설명 |
|---|---|
| `invoke()` | 하나의 입력 실행 |
| `batch()` | 여러 입력을 한 번에 실행 |
| `stream()` | 결과를 생성되는 대로 출력 |

즉, 작은 예제로 만든 체인을 대량 처리나 스트리밍으로 쉽게 확장할 수 있습니다.

# 3. Runnable 실행 방식

- Runnable은 LangChain에서 "실행 가능한 객체"
- Prompt, Model, Parser, 직접 만든 함수까지 모두 Runnable처럼 연결할 수 있음

## (1) `invoke`

가장 기본적인 단일 실행입니다.

## (2) `batch`

여러 입력을 같은 체인으로 한 번에 처리합니다. 고객 후기 분석, 문서 요약, 강의 주제별 설명 생성처럼 반복 작업에 유용합니다.

## (3) `stream`

`stream()`은 결과가 생성되는 동안 조각을 순서대로 받을 수 있습니다. 챗봇의 타이핑 효과처럼 사용자 경험을 좋게 만들 때 사용합니다.

# 4. Runnable 컴포넌트

LCEL 체인을 구성하는 모든 부품은 공통적으로 `Runnable` 인터페이스를 따릅니다. 보다 복잡한 처리 흐름을 구성할 때 다음 유틸리티 컴포넌트를 활용합니다.

| 컴포넌트 | 역할 |
|---|---|
| `RunnableSequence` | 파이프라인 (`\|` 연산자로 자동 생성) |
| `RunnableLambda` | 일반 파이썬 함수를 Runnable로 래핑 |
| `RunnableParallel` | 여러 체인을 병렬 실행 |

## 4.1. RunnableSequence
- LCEL의 가장 기본적인 구성

## 4.2. RunnableLambda

`RunnableLambda`는 일반 파이썬 함수를 LangChain 체인 안에 넣을 때 사용합니다.

예를 들어 모델 답변 뒤에 자동 안내 문구를 붙이거나, 출력 문자열을 후처리하거나, 입력값을 정리하는 함수를 체인에 연결할 수 있습니다.

### chain 데코레이터를 사용한 RunnableLambda 구현

### RunnableLambda 자동 변환


### Runnable의 입력 타입과 출력 타입에 주의


## [실습] 후처리 함수 만들기

모델 답변의 앞뒤 공백을 제거하고, 맨 앞에 `[강의 메모]`를 붙이는 함수를 작성해 봅니다.

In [ ]:
# def format_lecture_memo(text: str) -> str:
#     # 여기에 코드를 작성하세요.
#     return text
#
# memo_chain = concept_chain | RunnableLambda(format_lecture_memo)
# print(memo_chain.invoke({"concept": "RunnableLambda", "level": "입문"}))

## 4.3. RunnableParallel

`RunnableParallel`은 같은 입력을 여러 체인에 동시에 전달하고, 결과를 딕셔너리로 모아줍니다.

### RunnableParallel의 출력을 Runnable의 입력으로 연결하기


### RunnableParallel 자동 변환

## [실습] 주식 분석 리포트 생성기

이제 기본 파트의 Model, Prompt, Output Parser를 연결해 금융/주식 분석용 작은 프로젝트를 만듭니다.

이번 예시는 실시간 시세 조회나 투자 추천이 아니라, **제공된 예시 정보만 바탕으로 주식 분석 리포트 형식을 연습하는 실습**입니다.

## 목표

하나의 종목 또는 기업 정보를 입력하면 다음 내용을 구조화해서 생성합니다.

| 필드 | 설명 |
|---|---|
| `ticker` | 분석 대상 종목 또는 기업명 |
| `business_summary` | 기업과 업종에 대한 요약 |
| `investment_points` | 투자 관점에서 볼 만한 긍정 요인 |
| `risk_factors` | 확인해야 할 리스크 요인 |
| `key_indicators` | 분석할 때 참고할 핵심 지표 |
| `analyst_memo` | 투자 판단 전 확인해야 할 애널리스트 메모 |
